# 🛰️ Pipeline Completo — Monitoreo de Calidad del Agua
## Imágenes Sentinel-2/3 + Corrección ACOLITE + Modelo MDN

---

**Pipeline genérico y replicable** para cualquier cuerpo de agua, región geográfica y conjunto de estaciones de muestreo.

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | **Configuración** | Única celda que el usuario debe editar |
| 2 | **AOI automático** | Cálculo del área de interés desde las coordenadas del CSV |
| 3 | **Búsqueda** | Búsqueda de imágenes S2/S3 en Copernicus |
| 4 | **Descarga** | Descarga automatizada de imágenes satelitales |
| 5 | **Gestión de archivos** | Renombrado, descompresión y organización |
| 6 | **ACOLITE** | Corrección atmosférica y procesamiento masivo |
| 7 | **Extracción** | Extracción de Rrs y algoritmos desde NetCDF |
| 8 | **Diagnóstico** | Control de calidad y detección de errores |
| 9 | **CHL-MDN** | Estimación de clorofila con redes neuronales (MDN) |

> ⚠️ **Solo necesitas editar la Sección 1.** El resto del pipeline se adapta automáticamente a tu proyecto.

---
## 🔧 SECCIÓN 1 — Configuración del Entorno

### Instalación (ejecutar en terminal UNA sola vez)

```bash
# Opción A — virtualenv
python3.12 -m venv mi_entorno_aq
source mi_entorno_aq/bin/activate          # Linux / macOS
mi_entorno_aq\Scripts\activate             # Windows

# Opción B — conda
conda create -n mi_entorno_aq python=3.12
conda activate mi_entorno_aq

# Instalar dependencias
pip install pandas numpy scikit-learn netCDF4 requests scipy
pip install git+https://github.com/STREAM-RS/MDN-STREAM.git@development
```

📎 Referencia MDN: [github.com/ryan-edward-oshea/MDN_tutorials](https://github.com/ryan-edward-oshea/MDN_tutorials)

---
### ▶ Celda de configuración — **Editar solo aquí**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║        CONFIGURACIÓN GLOBAL — ÚNICA CELDA QUE DEBES EDITAR      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

# ─────────────────────────────────────────────────────────────────────────────
# 1. CREDENCIALES COPERNICUS
#    Regístrate gratis en: https://dataspace.copernicus.eu
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Pon tu email y contraseña de Copernicus Data Space.
#
#   ALTERNATIVA MÁS SEGURA — leer desde archivo .env:
#   1) Instala: pip install python-dotenv
#   2) Crea un archivo .env en la misma carpeta del notebook con:
#          COPERNICUS_USER=tu@email.com
#          COPERNICUS_PASS=tu_password
#   3) Descomenta las líneas del bloque 'dotenv' de abajo.
#   Esto evita que tus credenciales queden guardadas en el notebook.
USERNAME = "TU_EMAIL@ejemplo.com"
PASSWORD = "TU_PASSWORD"

# ── Descomenta para usar archivo .env en vez de escribir la contraseña aquí ──
# from dotenv import load_dotenv
# load_dotenv()
# USERNAME = os.getenv("COPERNICUS_USER")
# PASSWORD = os.getenv("COPERNICUS_PASS")

# ─────────────────────────────────────────────────────────────────────────────
# 2. ARCHIVO DE DATOS DE CAMPO
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Ruta completa a tu CSV de muestreos de campo.
ARCHIVO_CAMPO = r"D:\MiProyecto\datos_campo.csv"

# ▶ ADAPTAR: Nombres EXACTOS de las columnas en TU archivo CSV.
#   Si tu CSV se llama distinto (ej: 'Fecha', 'lat', 'lng', 'Chla'),
#   cámbialos aquí. El pipeline usará estas variables en TODAS las secciones.
COL_FECHA  = "date"       # Columna con la fecha (formato YYYY-MM-DD recomendado)
COL_LAT    = "Latitud"    # Columna de latitud decimal WGS84 (positivo = Norte)
COL_LON    = "Longitud"   # Columna de longitud decimal WGS84 (negativo = Oeste)
COL_SITIO  = "site"       # Columna con el nombre o ID de la estación de muestreo
COL_DATO   = "data"       # Columna con el valor medido in situ (ej. clorofila mg/m³)

# ─────────────────────────────────────────────────────────────────────────────
# 3. CARPETAS DE TRABAJO
#    Todas las carpetas se crean automáticamente si no existen.
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Carpeta raíz de tu proyecto.
CARPETA_RAIZ = r"D:\MiProyecto"

# ✦ AUTO: Las subcarpetas se construyen desde CARPETA_RAIZ.
#   Si necesitas separar las imágenes en otro disco por espacio,
#   puedes sobreescribir CARPETA_ZIPS o CARPETA_SAFE manualmente.
CARPETA_DATOS     = os.path.join(CARPETA_RAIZ, "datos")
CARPETA_ZIPS      = os.path.join(CARPETA_RAIZ, "imagenes_zip")    # ~700 MB por imagen S2
CARPETA_SAFE      = os.path.join(CARPETA_RAIZ, "imagenes_safe")   # Carpetas .SAFE/.SEN3
CARPETA_PROCESADO = os.path.join(CARPETA_RAIZ, "procesado_nc")    # NetCDF L2W de ACOLITE
CARPETA_CONFIGS   = os.path.join(CARPETA_RAIZ, "acolite_configs") # Settings temporales
CARPETA_SALIDAS   = os.path.join(CARPETA_RAIZ, "salidas")         # CSVs y reportes finales

# ─────────────────────────────────────────────────────────────────────────────
# 4. SENSORES A DESCARGAR
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Lista de sensores. Opciones: "S2" y/o "S3".
#   Solo S2:  SENSORES_BUSCAR = ["S2"]
#   Solo S3:  SENSORES_BUSCAR = ["S3"]
#   Ambos:    SENSORES_BUSCAR = ["S2", "S3"]
SENSORES_BUSCAR = ["S2", "S3"]

# ─────────────────────────────────────────────────────────────────────────────
# 5. ÁREA DE INTERÉS — SE CALCULA AUTOMÁTICAMENTE EN LA SECCIÓN 2
#    El pipeline lee las coordenadas de tu CSV y construye automáticamente
#    un polígono que cubre TODOS tus puntos + un margen configurable.
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR (opcional): Margen en grados alrededor del conjunto de estaciones.
#   0.05° ≈ 5 km | 0.1° ≈ 11 km | 0.5° ≈ 55 km
#   Aumentar si tus estaciones quedan muy cerca del borde de los granulos Sentinel.
BUFFER_GRADOS = 0.05

# ✦ AUTO: Estas variables se calculan en §2 y se usan en §3 y §6.
#   No las edites aquí; se llenan solas.
AOI_POLIGONO = None  # WKT POLYGON para la API de Copernicus
LIMIT_BOX    = None  # Formato ACOLITE: "sur,oeste,norte,este"

# ─────────────────────────────────────────────────────────────────────────────
# 6. RUTA DEL EJECUTABLE DE ACOLITE
#    Descarga desde: https://github.com/acolite/acolite/releases
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Ruta completa al ejecutable de ACOLITE.
#   Windows: termina en acolite.exe
#   Linux/macOS: termina en acolite (sin extensión)
ACOLITE_EXE = r"D:\herramientas\acolite_py_win_20251013.0\dist\acolite\acolite.exe"

# ─────────────────────────────────────────────────────────────────────────────
# 7. PARÁMETROS DE CORRECCIÓN ATMOSFÉRICA (ACOLITE)
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR (opcional): Resolución Sentinel-2 en metros. Opciones: 10, 20, 60.
#   20 m es un buen balance entre detalle y tamaño de archivo.
S2_RESOLUCION = "20"

# ▶ ADAPTAR (opcional): Productos de salida de ACOLITE.
#   Rrs_* = todas las bandas de reflectancia de agua.
#   Otros algoritmos disponibles: chl_oc3, chl_re_gons, chl_nn, spm_nechad2016,
#                                 tur_nechad2016, fai, chl_re_gons740
ACOLITE_PRODUCTOS = "Rrs_*,chl_oc3,chl_re_gons,spm_nechad2016,tur_nechad2016,fai"

# ▶ ADAPTAR (opcional): Umbral de máscara NIR para identificar agua.
#   Valores más bajos = criterio más estricto (más píxeles enmascarados).
#   0.055 es el valor estándar para aguas interiores y costeras.
ACOLITE_MASK_THRESHOLD = "0.055"

# ─────────────────────────────────────────────────────────────────────────────
# 8. CONFIGURACIÓN DEL MODELO MDN
#    Mixture Density Network para estimación de parámetros ópticos.
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Sensor compatible con MDN-STREAM.
#   Opciones comunes: "MSI" (S2), "OLCI" (S3), "OLI" (Landsat-8/9), "MODIS", "MERIS"
#   Lista completa: https://github.com/STREAM-RS/MDN-STREAM
MDN_SENSOR = "MSI"

# ▶ ADAPTAR: Variable que el modelo debe estimar.
#   Opciones típicas: "chl" (clorofila-a), "tss" (sólidos suspendidos),
#                     "cdom" (materia orgánica disuelta), "pc" (ficocinanina)
MDN_PRODUCTO = "chl"

# ▶ ADAPTAR (opcional): UID del modelo pre-entrenado específico.
#   Dejar vacío ("") para usar el modelo por defecto del sensor.
#   Útil para reproducir resultados exactos con un modelo publicado.
MDN_MODEL_UID = "1b79098defa9693eb03ac734467c3854cf4fb81ce3d1a96b9bf0f511f1c997b1"

# ─────────────────────────────────────────────────────────────────────────────
# 9. MAPEO DE BANDAS Rrs  (nombre_columna_salida → longitudes_de_onda)
#    Sentinel-2A y 2B tienen longitudes de onda centrales ligeramente distintas.
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Si usas Sentinel-3 OLCI o Landsat, cambia los valores de onda.
#   Formato: "Nombre_columna": ["onda_sensor_variante_A", "onda_sensor_variante_B"]
#   Para un solo tipo de sensor: duplica el valor: ["443", "443"]
MAPEO_RRS = {
    "Rrs_443": ["443", "442"],  # Azul profundo — aerosoles costeros
    "Rrs_492": ["492", "492"],  # Azul
    "Rrs_560": ["560", "559"],  # Verde — sensible a clorofila
    "Rrs_665": ["665", "665"],  # Rojo
    "Rrs_704": ["704", "704"],  # Red edge 1 — clorofila alta
    "Rrs_740": ["740", "739"],  # Red edge 2
    "Rrs_783": ["783", "780"],  # Red edge 3
    "Rrs_833": ["833", "833"],  # NIR
    "Rrs_865": ["865", "864"],  # NIR amplio
}

# ▶ ADAPTAR (opcional): Algoritmos adicionales a extraer de los NetCDF de ACOLITE.
#   Los nombres deben coincidir exactamente con las variables en los archivos .nc.
ALGOS_ACOLITE = [
    "chl_oc3",        # Clorofila-a (band ratio azul/verde)
    "chl_re_gons",    # Clorofila-a (red edge, Gons et al.)
    "chl_re_gons740", # Variante Gons con banda 740 nm
    "chl_nn",         # Clorofila-a (red neuronal ACOLITE)
    "tur_nechad2016", # Turbidez (Nechad et al. 2016)
    "fai",            # Floating Algae Index
    "spm_nechad2016", # Sólidos suspendidos (Nechad et al. 2016)
]

# ─────────────────────────────────────────────────────────────────────────────
# 10. NOMBRE DEL PROYECTO
#     Se usa para nombrar todos los archivos de salida y logs.
#     Sin espacios ni caracteres especiales.
# ─────────────────────────────────────────────────────────────────────────────
# ▶ ADAPTAR: Nombre corto y descriptivo de tu proyecto.
NOMBRE_PROYECTO = "MiProyectoAcuatico"

# ═══════════════════════════════════════════════════════════════════════════════
# ✦ AUTO: Rutas de salida — se construyen automáticamente. NO editar.
#   Cada proyecto genera sus propios archivos sin pisar los de otros proyectos.
# ═══════════════════════════════════════════════════════════════════════════════
for _c in [CARPETA_DATOS, CARPETA_ZIPS, CARPETA_SAFE,
           CARPETA_PROCESADO, CARPETA_CONFIGS, CARPETA_SALIDAS]:
    os.makedirs(_c, exist_ok=True)

ARCHIVO_DESCARGA = os.path.join(CARPETA_DATOS,   f"{NOMBRE_PROYECTO}_lista_descarga.csv")
ARCHIVO_MATRIZ   = os.path.join(CARPETA_SALIDAS, f"{NOMBRE_PROYECTO}_Rrs_Algoritmos.csv")
ARCHIVO_FINAL    = os.path.join(CARPETA_SALIDAS, f"{NOMBRE_PROYECTO}_Resultados_MDN.csv")
ARCHIVO_LOG      = os.path.join(CARPETA_SALIDAS, f"{NOMBRE_PROYECTO}_extraccion.log")
ARCHIVO_STATS    = os.path.join(CARPETA_SALIDAS, f"{NOMBRE_PROYECTO}_estadisticas.csv")
ARCHIVO_REPORTE  = os.path.join(CARPETA_SALIDAS, f"{NOMBRE_PROYECTO}_diagnostico.csv")
ARCHIVO_BAT      = os.path.join(CARPETA_RAIZ,    f"{NOMBRE_PROYECTO}_acolite.bat")

print("✅ Configuración cargada.")
print(f"   Proyecto  : {NOMBRE_PROYECTO}")
print(f"   Raíz      : {CARPETA_RAIZ}")
print(f"   Sensor MDN: {MDN_SENSOR} → {MDN_PRODUCTO}")
print(f"   Sensores  : {', '.join(SENSORES_BUSCAR)}")

---
## 📐 SECCIÓN 2 — Cálculo Automático del Área de Interés (AOI)

Lee las coordenadas de **todas** tus estaciones de muestreo y construye automáticamente:
- `AOI_POLIGONO` — Polígono WKT para la búsqueda en Copernicus (cubre todos los puntos)
- `LIMIT_BOX` — Recorte espacial para ACOLITE (`sur,oeste,norte,este`)

Ambos incluyen el margen `BUFFER_GRADOS` definido en §1.

In [ ]:
# ── Cálculo automático del AOI desde el CSV de campo ─────────────────────────
import pandas as pd

# ✦ AUTO: Leer el CSV usando COL_LAT y COL_LON de la configuración.
#   No importa cómo se llamen las columnas en tu archivo; el pipeline
#   siempre usará los nombres que mapeaste en §1.
try:
    df_campo = pd.read_csv(ARCHIVO_CAMPO)
    print(f"📂 CSV cargado: {len(df_campo)} registros")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\n❌ No se encontró: {ARCHIVO_CAMPO}"
        "\n   Revisa la variable ARCHIVO_CAMPO en la Sección 1."
    )

# Verificar que las columnas configuradas existen en el CSV
cols_req     = [COL_FECHA, COL_LAT, COL_LON, COL_SITIO, COL_DATO]
cols_missing = [c for c in cols_req if c not in df_campo.columns]
if cols_missing:
    raise ValueError(
        f"\n❌ Columnas no encontradas: {cols_missing}"
        f"\n   Columnas disponibles en el CSV: {list(df_campo.columns)}"
        "\n   Ajusta los valores COL_* en la Sección 1."
    )

# ✦ AUTO: Calcular el bounding box que abarca TODAS las estaciones de muestreo.
#   Esto garantiza que la imagen descargada cubra cada punto del estudio,
#   sin importar cuántos puntos haya ni cuán dispersos estén.
lat_min = df_campo[COL_LAT].min() - BUFFER_GRADOS   # Límite Sur
lat_max = df_campo[COL_LAT].max() + BUFFER_GRADOS   # Límite Norte
lon_min = df_campo[COL_LON].min() - BUFFER_GRADOS   # Límite Oeste
lon_max = df_campo[COL_LON].max() + BUFFER_GRADOS   # Límite Este

# ✦ AUTO: Construir el polígono WKT para la API OData de Copernicus.
#   Se usa POLYGON en vez de POINT para asegurar que la imagen encontrada
#   cubra TODOS los puntos, no solo uno central arbitrario.
AOI_POLIGONO = (
    f"POLYGON(({lon_min:.6f} {lat_min:.6f}, "
    f"{lon_max:.6f} {lat_min:.6f}, "
    f"{lon_max:.6f} {lat_max:.6f}, "
    f"{lon_min:.6f} {lat_max:.6f}, "
    f"{lon_min:.6f} {lat_min:.6f}))"
)

# ✦ AUTO: Formato LIMIT_BOX para ACOLITE → "sur,oeste,norte,este".
#   Recorta las imágenes exactamente al área de interés, lo que reduce
#   el tamaño de los archivos NetCDF y acelera el procesamiento.
LIMIT_BOX = f"{lat_min:.4f},{lon_min:.4f},{lat_max:.4f},{lon_max:.4f}"

print(f"\n📍 Estaciones únicas : {df_campo[COL_SITIO].nunique()}")
print(f"   Fechas únicas     : {pd.to_datetime(df_campo[COL_FECHA], errors='coerce').dt.date.nunique()}")
print(f"\n🗺️  Bounding Box (buffer = {BUFFER_GRADOS}°):")
print(f"   Norte  {lat_max:.4f}°  |  Sur   {lat_min:.4f}°")
print(f"   Este   {lon_max:.4f}°  |  Oeste {lon_min:.4f}°")
print(f"\n   AOI      : {AOI_POLIGONO}")
print(f"   LIMIT_BOX: {LIMIT_BOX}")
print("\n✅ AOI calculado y listo.")

---
## 🔍 SECCIÓN 3 — Búsqueda de Imágenes en Copernicus

Consulta el catálogo **CDSE** vía API OData para cada fecha de muestreo.  
Busca en el área calculada automáticamente y solo los sensores en `SENSORES_BUSCAR`.

**Salida:** `lista_descarga.csv` — `Fecha | S2_ID | S3_ID`

In [ ]:
# ── Búsqueda de imágenes en el catálogo Copernicus ────────────────────────────
import pandas as pd, requests, time

# ✦ AUTO: Leer fechas desde el CSV usando COL_FECHA de la configuración.
df_campo     = pd.read_csv(ARCHIVO_CAMPO)
unique_dates = sorted([
    d for d in pd.to_datetime(df_campo[COL_FECHA], errors='coerce').dt.date.unique()
    if pd.notna(d)
])

if not unique_dates:
    raise ValueError(
        f"❌ Sin fechas válidas en columna '{COL_FECHA}'.\n"
        "   Verifica el nombre de la columna y el formato (YYYY-MM-DD recomendado)."
    )
print(f"📅 Fechas a buscar: {len(unique_dates)}")

BASE_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

# ✦ AUTO: Tipos de producto Copernicus por sensor.
#   Si en el futuro cambian los nombres, edita solo este diccionario.
TIPOS_PRODUCTO = {
    "S2": "S2MSI1C",      # Sentinel-2 Level-1C (Top of Atmosphere)
    "S3": "OL_1_EFR___",  # Sentinel-3 OLCI Full Resolution
}

matchup_list = []
print(f"\n🔎 Buscando con AOI bounding box (sensores: {', '.join(SENSORES_BUSCAR)})...\n")

for date in unique_dates:
    start = f"{date}T00:00:00.000Z"
    end   = f"{date}T23:59:59.999Z"
    res   = {"Fecha": date, "S2_ID": None, "S3_ID": None}

    for sensor in ["S2", "S3"]:
        # ✦ AUTO: Solo consultar los sensores en SENSORES_BUSCAR.
        if sensor not in SENSORES_BUSCAR:
            continue

        # ✦ AUTO: Usar AOI_POLIGONO (calculado en §2) para cubrir todos los puntos.
        #   Esto reemplaza el POINT fijo del código original, que solo garantizaba
        #   cobertura de un único punto central, posiblemente sin cubrir estaciones
        #   en los extremos del área de estudio.
        filtro = (
            f"Attributes/OData.CSC.StringAttribute/any("
            f"att:att/Name eq 'productType' and att/Value eq '{TIPOS_PRODUCTO[sensor]}') "
            f"and ContentDate/Start ge {start} and ContentDate/Start le {end} "
            f"and OData.CSC.Intersects(area=geography'SRID=4326;{AOI_POLIGONO}')"
        )

        try:
            r    = requests.get(BASE_URL, params={"$filter": filtro, "$top": 1}, timeout=30)
            data = r.json()
            if data.get('value'):
                res[f"{sensor}_ID"] = data['value'][0]['Id']
        except Exception as e:
            print(f"  ⚠️  Error {sensor} en {date}: {e}")

    s2 = "✅" if res["S2_ID"] else ("──" if "S2" in SENSORES_BUSCAR else "N/A")
    s3 = "✅" if res["S3_ID"] else ("──" if "S3" in SENSORES_BUSCAR else "N/A")
    print(f"  {date}  →  S2: {s2}  |  S3: {s3}")
    matchup_list.append(res)
    time.sleep(0.3)   # Respetar límites de la API

df_desc = pd.DataFrame(matchup_list)
df_desc.to_csv(ARCHIVO_DESCARGA, index=False)

print(f"\n{'='*50}")
for s in ["S2", "S3"]:
    if s in SENSORES_BUSCAR:
        n = df_desc[f"{s}_ID"].notna().sum()
        print(f"  {s} encontradas: {n} / {len(unique_dates)} fechas")
print(f"  💾 {ARCHIVO_DESCARGA}")

---
## ⬇️ SECCIÓN 4 — Descarga de Imágenes Satelitales

Descarga con renovación automática de token JWT (expira cada ~10 min).  
Omite archivos ya descargados y verifica integridad por tamaño.

> 💡 **Espacio requerido estimado:** ~700 MB por imagen S2 · ~350 MB por S3

In [ ]:
# ── Funciones reutilizables: autenticación y descarga ─────────────────────────
import requests, os

AUTH_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

def get_token():
    """Obtiene un token Bearer de Copernicus CDSE.
    Usa USERNAME y PASSWORD de la Sección 1 (configuración global).
    """
    try:
        r = requests.post(AUTH_URL, data={
            "client_id": "cdse-public", "grant_type": "password",
            "username": USERNAME, "password": PASSWORD
        }, timeout=30)
        if r.status_code == 200:
            return r.json()['access_token']
        print(f"[!] Error HTTP {r.status_code}: {r.text[:300]}")
    except Exception as e:
        print(f"[!] Error de conexión: {e}")
    return None


def download_product(product_id, token, filename, output_folder):
    """Descarga un producto Sentinel a output_folder.
    - Omite si ya existe y tiene tamaño válido (> 1 KB).
    - Elimina archivos corruptos (< 1 KB) antes de reintentar.
    - Renueva el token automáticamente ante error 401 (expirado).
    Retorna: (éxito: bool, token_vigente: str)
    """
    url  = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    path = os.path.join(output_folder, f"{filename}.zip")

    if os.path.exists(path):
        if os.path.getsize(path) > 1000:
            print(f"   [SKIP] {filename} ya existe.")
            return True, token
        os.remove(path)   # Borrar corrupto y re-descargar
        print(f"   [INFO] Corrupto eliminado, re-descargando: {filename}")

    def _descargar(tok):
        with requests.get(url, headers={"Authorization": f"Bearer {tok}"},
                          stream=True, timeout=300) as r:
            if r.status_code == 200:
                with open(path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                return True
            return r.status_code

    resultado = _descargar(token)
    if resultado == 401:   # Token expirado → renovar y reintentar
        print("   [AVISO] Token expirado. Renovando...")
        nuevo = get_token()
        if nuevo and _descargar(nuevo) is True:
            return True, nuevo
        return False, token
    elif resultado is True:
        return True, token
    print(f"   [ERROR] HTTP {resultado} — {filename}")
    return False, token


print("✅ Funciones de autenticación y descarga listas.")

In [ ]:
# ── Descarga de todos los sensores en SENSORES_BUSCAR ─────────────────────────
import pandas as pd, time

try:
    df_manifest = pd.read_csv(ARCHIVO_DESCARGA)
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Ejecuta primero la Sección 3 (Búsqueda): {ARCHIVO_DESCARGA}")

token = get_token()
if not token:
    raise RuntimeError("❌ No se pudo iniciar sesión. Verifica tus credenciales en §1.")

# ✦ AUTO: El bucle itera sobre SENSORES_BUSCAR — no hay código duplicado
#   para S2 y S3. Si agregas un tercer sensor al config, se descarga solo.
for sensor in SENSORES_BUSCAR:
    col_id = f"{sensor}_ID"
    if col_id not in df_manifest.columns:
        print(f"⚠️  Sin columna '{col_id}' en el manifiesto. Saltando {sensor}.")
        continue

    df_s = df_manifest[df_manifest[col_id].notna()]
    print(f"\n{'━'*55}\n  {sensor}: {len(df_s)} imágenes a descargar\n{'━'*55}")

    for i, (_, row) in enumerate(df_s.iterrows(), 1):
        # ✦ AUTO: El nombre del archivo usa NOMBRE_PROYECTO para evitar colisiones
        #   entre proyectos que guarden imágenes en la misma carpeta.
        nombre = f"{sensor}_{str(row['Fecha']).replace('-', '_')}"
        print(f"[{i}/{len(df_s)}] {nombre}")
        _, token = download_product(row[col_id], token, nombre, CARPETA_ZIPS)
        time.sleep(1)

print("\n✅ Descarga completada.")

In [ ]:
# ── Descarga puntual: re-descargar una imagen específica por nombre ────────────
# ▶ ADAPTAR: Sustituye TARGET_NAME_PART por la parte identificadora de
#   la imagen que necesitas (fecha, órbita, tile, etc.).
TARGET_NAME_PART = "S2A_MSIL1C_20200315"   # ← Editar según necesidad

def buscar_y_descargar_puntual(nombre_parcial):
    """Busca una imagen por nombre parcial y la descarga a CARPETA_ZIPS."""
    token = get_token()
    if not token:
        return
    print(f"🔍 Buscando imagen con: '{nombre_parcial}'")
    r = requests.get(
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Products",
        params={"$filter": f"contains(Name, '{nombre_parcial}')", "$top": 1},
        timeout=30
    ).json()

    if not r.get('value'):
        print("❌ Imagen no encontrada. Verifica el nombre.")
        return

    p = r['value'][0]
    print(f"✅ {p['Name']}  ({p['ContentLength']/1e6:.0f} MB)")
    download_product(p['Id'], token, p['Name'], CARPETA_ZIPS)
    print(f"💾 Guardada en: {CARPETA_ZIPS}")

# Descomenta para ejecutar:
# buscar_y_descargar_puntual(TARGET_NAME_PART)

---
## 📁 SECCIÓN 5 — Gestión de Archivos

1. **5.1 Renombrado** — Corrige nombres de ZIPs usando el nombre interno Sentinel
2. **5.2 Descompresión** — Extrae `.SAFE` / `.SEN3` y elimina ZIPs
3. **5.3 Lista de rutas** — Genera el listado para ACOLITE

In [ ]:
# ── 5.1 Renombrado de ZIPs ────────────────────────────────────────────────────
# Copernicus puede descargar ZIPs con nombres genéricos.
# El nombre correcto de Sentinel está dentro del ZIP como carpeta raíz.
import os, zipfile

print(f"🔧 Reparando nombres en: {CARPETA_ZIPS}")

for filename in os.listdir(CARPETA_ZIPS):
    if not filename.endswith(".zip"):
        continue
    file_path = os.path.join(CARPETA_ZIPS, filename)
    try:
        with zipfile.ZipFile(file_path, 'r') as zf:
            names = zf.namelist()
            if not names:
                print(f"  ⚠️  {filename}: vacío")
                continue
            # ✦ AUTO: La carpeta raíz es el nombre real del producto Sentinel.
            #   Funciona para S1/S2 (.SAFE) y S3 (.SEN3) sin distinción.
            root = names[0].split('/')[0]
            if any(root.startswith(p) for p in ["S1","S2","S3"]) and len(root) > 10:
                target = root + ".zip"
                if filename != target:
                    dest = os.path.join(CARPETA_ZIPS, target)
                    if not os.path.exists(dest):
                        os.rename(file_path, dest)
                        print(f"  ✅ {filename} → {target}")
                    else:
                        print(f"  ⏭️  Omitido: {target} ya existe")
                else:
                    print(f"  ✅ {filename}: nombre correcto")
            else:
                print(f"  ⚠️  {filename}: raíz inesperada '{root}'")
    except zipfile.BadZipFile:
        print(f"  ❌ {filename}: CORRUPTO — re-descargar con §4")
    except Exception as e:
        print(f"  ❌ {filename}: {e}")

print("\n✅ Renombrado terminado.")

In [ ]:
# ── 5.2 Descompresión y eliminación de ZIPs ───────────────────────────────────
import os, zipfile

def descomprimir_y_eliminar():
    zips  = [f for f in os.listdir(CARPETA_ZIPS) if f.endswith('.zip')]
    total = len(zips)
    print(f"📦 ZIPs a extraer: {total}  →  destino: {CARPETA_SAFE}")
    for i, nombre_zip in enumerate(zips, 1):
        ruta = os.path.join(CARPETA_ZIPS, nombre_zip)
        try:
            print(f"[{i}/{total}] Extrayendo: {nombre_zip}")
            with zipfile.ZipFile(ruta, 'r') as z:
                z.extractall(CARPETA_SAFE)
            os.remove(ruta)   # Borrar ZIP solo si extracción fue exitosa
            print(f"        ✅ Extraído y ZIP eliminado.")
        except zipfile.BadZipFile:
            print(f"        ❌ CORRUPTO — ZIP conservado: {nombre_zip}")
        except Exception as e:
            print(f"        ❌ {e} — ZIP conservado.")
    print(f"\n✅ Extracción completada.")

conf = input("⚠️  ¿Eliminar ZIPs originales tras extraer? (si/no): ")
if conf.strip().lower() == 'si':
    descomprimir_y_eliminar()
else:
    print("⏹️  Cancelado — ZIPs conservados.")

In [ ]:
# ── 5.3 Lista de rutas de imágenes para ACOLITE ───────────────────────────────
import os

# ✦ AUTO: Detecta .SAFE (Sentinel-2) y .SEN3 (Sentinel-3) automáticamente.
carpetas = [
    os.path.join(CARPETA_SAFE, d)
    for d in os.listdir(CARPETA_SAFE)
    if d.endswith(('.SAFE', '.SEN3'))
]

LISTA_IMAGENES = os.path.join(CARPETA_RAIZ, f"{NOMBRE_PROYECTO}_lista_imagenes.txt")

if not carpetas:
    print(f"⚠️  Sin carpetas .SAFE/.SEN3 en: {CARPETA_SAFE}")
else:
    with open(LISTA_IMAGENES, "w") as f:
        for c in carpetas:
            f.write(c + "\n")
    n_s2 = sum(1 for d in carpetas if os.path.basename(d).startswith(('S2A','S2B')))
    n_s3 = sum(1 for d in carpetas if os.path.basename(d).startswith('S3'))
    print(f"✅ {len(carpetas)} imágenes → {LISTA_IMAGENES}")
    print(f"   Sentinel-2: {n_s2} | Sentinel-3: {n_s3}")

---
## 🌍 SECCIÓN 6 — Corrección Atmosférica (ACOLITE)

Genera el script `.bat` para procesar todas las imágenes con ACOLITE  
(Dark Spectrum Fitting). El recorte espacial usa `LIMIT_BOX` calculado en §2.

In [ ]:
# ── 6.1 Generación del script BAT para ACOLITE ────────────────────────────────
import os

# ✦ AUTO: Detectar imágenes .SAFE disponibles en CARPETA_SAFE.
imagenes_s2 = [f for f in os.listdir(CARPETA_SAFE) if f.endswith(".SAFE")]

if not imagenes_s2:
    print("⚠️  Sin carpetas .SAFE. Ejecuta la Sección 5 primero.")
else:
    # ✦ AUTO: Todos los parámetros vienen de la Sección 1 de configuración.
    #   El LIMIT_BOX se calculó automáticamente en §2 a partir de las coordenadas del CSV.
    ACOLITE_PARAMS = {
        "s2_target_res":                    S2_RESOLUCION,
        "l2w_parameters":                   ACOLITE_PRODUCTOS,
        "rgb_rhot":                         "False",
        "rgb_rhos":                         "True",
        "map_l2w":                          "False",
        "dsf_aot_estimate":                 "tiled",
        "dsf_residual_glint_correction":    "True",
        "atmospheric_correction":           "True",
        "atmospheric_correction_method":    "dark_spectrum",
        "ancillary_data":                   "True",
        "l2w_data_in_memory":               "True",
        "l2w_mask_threshold":               ACOLITE_MASK_THRESHOLD,
        "l2w_mask_water_parameters":        "True",
        "delete_extracted_input":           "True",
        "l1r_delete_netcdf":                "True",
        "l2r_delete_netcdf":                "True",
        "delete_acolite_run_text_files":    "True",
        # ✦ AUTO: LIMIT_BOX calculado en §2 desde las coordenadas del CSV + buffer.
        "limit":                            LIMIT_BOX,
        "merge_tiles":                      "True",
        "merge_full_tiles":                 "True",
        "s2_include_all_granules":          "True",
        "s2_time_difference_check":         "999999",
        "s2_granule_time_difference_check": "999999",
    }

    with open(ARCHIVO_BAT, "w") as bat:
        bat.write("@echo off\n")
        bat.write(f"echo Procesando {NOMBRE_PROYECTO} con ACOLITE ({len(imagenes_s2)} imagenes)...\n\n")

        for i, img in enumerate(imagenes_s2):
            # ✦ AUTO: Detectar S2A o S2B desde el nombre de la imagen.
            sensor        = "S2A" if "S2A" in img else "S2B"
            settings_file = os.path.join(CARPETA_CONFIGS, f"settings_{i:04d}.txt")

            with open(settings_file, "w") as f:
                f.write(f"inputfile={os.path.join(CARPETA_SAFE, img)}\n")
                f.write(f"output={CARPETA_PROCESADO}\n")
                f.write(f"sensor={sensor}\n")
                for k, v in ACOLITE_PARAMS.items():
                    f.write(f"{k}={v}\n")

            bat.write(f"echo [{i+1}/{len(imagenes_s2)}] {img}\n")
            bat.write(f'"{ACOLITE_EXE}" --cli --settings="{settings_file}"\n')

        bat.write("echo === COMPLETADO ===\npause\n")

    print(f"✅ {len(imagenes_s2)} configuraciones generadas")
    print(f"   Ejecuta: {ARCHIVO_BAT}")

In [ ]:
# ── 6.2 Inspección de variables en los NetCDF generados ───────────────────────
# Si los nombres Rrs no coinciden con MAPEO_RRS, actualiza ese diccionario en §1.
import netCDF4 as nc, os

archivos_nc = [f for f in os.listdir(CARPETA_PROCESADO) if f.endswith("_L2W.nc")][:5]
if not archivos_nc:
    print("⚠️  Sin archivos _L2W.nc. Ejecuta ACOLITE primero.")
else:
    print(f"🔬 Variables Rrs en primeros {len(archivos_nc)} NetCDF:\n")
    for arch in archivos_nc:
        print(f"  📄 {arch}")
        with nc.Dataset(os.path.join(CARPETA_PROCESADO, arch)) as ds:
            for v in sorted(v for v in ds.variables
                            if any(p in v.lower() for p in ['rrs','rhow','rhos'])):
                print(f"     • {v}")

---
## 📊 SECCIÓN 7 — Extracción de Rrs y Algoritmos desde NetCDF

Empareja cada punto de muestreo con su imagen y extrae reflectancias y algoritmos.  
Usa las columnas `COL_*` configuradas en §1 — funciona con cualquier CSV.

In [ ]:
# ── Funciones de extracción espacial ─────────────────────────────────────────
import netCDF4 as nc, numpy as np, pandas as pd
from collections import defaultdict

# ✦ AUTO: PREFIJOS_RRS cubre los distintos nombres que ACOLITE puede usar.
PREFIJOS_RRS = ['Rrs_', 'rrs_', 'rhow_', 'rhos_']

def buscar_variable_rrs(ds, wave):
    """Busca una variable Rrs probando distintos prefijos y grafías."""
    for p in PREFIJOS_RRS:
        if f"{p}{wave}" in ds.variables:
            return f"{p}{wave}"
    for p in PREFIJOS_RRS:
        for v in ds.variables:
            if v.startswith(p) and v.replace(p,'') == str(wave):
                return v
    return None

def extraer_ventana(data, py, px, var_name="", radio=1):
    """Promedio de una ventana (2r+1)×(2r+1) con filtros de rango físico."""
    try:
        v    = data[py-radio:py+radio+1, px-radio:px+radio+1]
        vals = v.compressed() if np.ma.is_masked(v) else v.flatten()
        vals = vals[~np.isnan(vals)]
        vn   = var_name.lower()
        if len(vals) > 0:
            if any(p in vn for p in ['rrs','rhow','rhos']):
                vals = vals[(vals >= -0.01) & (vals <= 0.5)]
            elif 'chl' in vn:
                vals = vals[(vals >= 0) & (vals <= 500)]
            elif 'spm' in vn or 'tur' in vn:
                vals = vals[(vals >= 0) & (vals <= 5000)]
        return (float(np.mean(vals)), len(vals)) if len(vals) > 0 else (np.nan, 0)
    except:
        return np.nan, 0

def extraer_rrs(ds, py, px, wave):
    var = buscar_variable_rrs(ds, wave)
    if var is None: return np.nan, None, 0
    try:
        v, n = extraer_ventana(ds.variables[var][:], py, px, var)
        return v, var, n
    except: return np.nan, var, 0

def extraer_algo(ds, py, px, var_name):
    if var_name not in ds.variables: return np.nan, False, 0
    try:
        v, n = extraer_ventana(ds.variables[var_name][:], py, px, var_name)
        return v, True, n
    except: return np.nan, True, 0

def coordenadas_validas(py, px, shape, m=2):
    my, mx = shape
    return m <= py < my-m and m <= px < mx-m

def distancia_m(lat1, lon1, lat2, lon2):
    dy = (lat1-lat2)*111320
    dx = (lon1-lon2)*111320*np.cos(np.radians((lat1+lat2)/2))
    return float(np.sqrt(dy**2+dx**2))

print("✅ Funciones de extracción cargadas.")

In [ ]:
# ── 7.1 Extractor principal — ventana 3×3 ────────────────────────────────────
import os

# ✦ AUTO: Usa COL_* para leer el CSV; compatible con cualquier estructura.
df_campo = pd.read_csv(ARCHIVO_CAMPO)
df_campo['date_std'] = pd.to_datetime(df_campo[COL_FECHA], errors='coerce').dt.strftime('%Y_%m_%d')
df_campo = df_campo.dropna(subset=['date_std', COL_LAT, COL_LON])

archivos_nc = [f for f in os.listdir(CARPETA_PROCESADO) if f.endswith("_L2W.nc")]
if not archivos_nc:
    raise FileNotFoundError(f"❌ Sin NetCDF en {CARPETA_PROCESADO}. Ejecuta ACOLITE.")

# Índice rápido fecha → archivos NC
idx_fecha = defaultdict(list)
for arch in archivos_nc:
    p = arch.split('_')
    for i in range(len(p)-2):
        if p[i].isdigit() and len(p[i]) == 4:
            idx_fecha[f"{p[i]}_{p[i+1]}_{p[i+2]}"].append(arch)
            break

stats, matriz = defaultdict(int), []
print(f"🛰️  Procesando {len(df_campo)} puntos...\n")

for _, row in df_campo.iterrows():
    stats['total'] += 1
    imgs = idx_fecha.get(row['date_std'], [])
    if not imgs:
        stats['sin_imagen'] += 1
        continue

    for arch in imgs:
        try:
            with nc.Dataset(os.path.join(CARPETA_PROCESADO, arch), 'r') as ds:
                lats = ds.variables['lat'][:]
                lons = ds.variables['lon'][:]
                dm   = (lats - row[COL_LAT])**2 + (lons - row[COL_LON])**2
                py, px = np.unravel_index(np.argmin(dm), dm.shape)

                if not coordenadas_validas(py, px, lats.shape):
                    stats['fuera_rango'] += 1; continue

                dist = distancia_m(row[COL_LAT], row[COL_LON],
                                   float(lats[py,px]), float(lons[py,px]))

                # ✦ AUTO: Detectar sensor desde el nombre del archivo.
                if   "S2A" in arch: sensor, s_idx = "S2A", 0
                elif "S2B" in arch: sensor, s_idx = "S2B", 1
                else:               sensor, s_idx = "S3",  0

                # ✦ AUTO: Construir resultado usando COL_* de la config.
                res = {
                    COL_SITIO:    row[COL_SITIO],
                    COL_FECHA:    row[COL_FECHA],
                    COL_LAT:      row[COL_LAT],
                    COL_LON:      row[COL_LON],
                    "Lat_Pixel":  float(lats[py,px]),
                    "Lon_Pixel":  float(lons[py,px]),
                    "Distancia_m": round(dist, 2),
                    COL_DATO:     row[COL_DATO],
                    "sensor":     sensor, "archivo_nc": arch,
                }

                ok = 0
                # ✦ AUTO: Extraer algoritmos desde ALGOS_ACOLITE de la config §1.
                for algo in ALGOS_ACOLITE:
                    v, found, _ = extraer_algo(ds, py, px, algo)
                    res[algo.upper()] = v
                    if found and not pd.isna(v): ok += 1

                # ✦ AUTO: Extraer Rrs usando MAPEO_RRS de la config §1.
                #   s_idx selecciona la longitud de onda correcta para S2A o S2B.
                for col_r, ondas in MAPEO_RRS.items():
                    v, vn, _ = extraer_rrs(ds, py, px, ondas[s_idx])
                    res[col_r] = v
                    if vn and not pd.isna(v): ok += 1

                if ok == 0:
                    stats['nublados'] += 1; continue

                matriz.append(res)
                stats['exitosos'] += 1
                print(f"  ✅ {row[COL_SITIO]} ({row[COL_FECHA]}) | {sensor} | {ok} bandas | {dist:.0f} m")
                break
        except Exception as e:
            print(f"  ❌ {arch}: {e}")
            stats['errores'] += 1

pd.DataFrame(matriz).to_csv(ARCHIVO_MATRIZ, index=False)
print(f"\n{'='*50}")
for k, v in stats.items(): print(f"  {k:15}: {v}")
print(f"  💾 {ARCHIVO_MATRIZ}")

In [ ]:
# ── 7.2 Extractor inteligente — búsqueda expandida (11×11 px) ────────────────
# Si la ventana 3×3 está enmascarada, amplía la búsqueda al píxel de agua
# válido más cercano dentro de un radio configurable.

# ▶ ADAPTAR (opcional): Radio de búsqueda en píxeles.
#   radio=5 → ventana 11×11 | radio=10 → ventana 21×21
#   Aumentar si tus estaciones están muy cerca de orillas o costas.
RADIO_EXPANDIDO = 5

ARCHIVO_VALIDACION = ARCHIVO_MATRIZ.replace(".csv", "_expandido.csv")

def extraer_inteligente(ds, lat_t, lon_t, var_name, radio=RADIO_EXPANDIDO):
    """Extrae con búsqueda progresiva: 3×3 → radio expandido."""
    if var_name not in ds.variables: return np.nan, 0, np.nan
    lats = ds.variables['lat'][:]
    lons = ds.variables['lon'][:]
    dm   = (lats - lat_t)**2 + (lons - lon_t)**2
    py, px = np.unravel_index(np.argmin(dm), dm.shape)

    v3, n3 = extraer_ventana(ds.variables[var_name][:], py, px, var_name, radio=1)
    if n3 > 0: return v3, n3, 0.0

    # Búsqueda expandida
    vb  = ds.variables[var_name][py-radio:py+radio+1, px-radio:px+radio+1]
    lb  = lats[py-radio:py+radio+1, px-radio:px+radio+1]
    ob  = lons[py-radio:py+radio+1, px-radio:px+radio+1]
    mk  = (~vb.mask if np.ma.is_masked(vb)
           else ~np.isnan(vb.data if hasattr(vb,'data') else vb))
    ys, xs = np.where(mk)
    if len(ys) > 0:
        ds_  = np.sqrt((lb[ys,xs]-lat_t)**2 + (ob[ys,xs]-lon_t)**2)
        idx  = np.argmin(ds_)
        return float(vb[ys[idx],xs[idx]]), 1, float(ds_[idx]*111320)
    return np.nan, 0, np.nan

# Ejecución
df_campo = pd.read_csv(ARCHIVO_CAMPO)
df_campo['date_std'] = pd.to_datetime(df_campo[COL_FECHA], errors='coerce').dt.strftime('%Y_%m_%d')
archivos_disco = [f for f in os.listdir(CARPETA_PROCESADO) if f.endswith("_L2W.nc")]
VAR_DIAG = ALGOS_ACOLITE[0]   # ✦ AUTO: Primer algoritmo como control de agua

matriz_val = []
print(f"🔍 Extracción expandida (radio={RADIO_EXPANDIDO}px)...")

for _, row in df_campo.iterrows():
    match = [f for f in archivos_disco if row['date_std'] in f]
    if not match: continue
    try:
        with nc.Dataset(os.path.join(CARPETA_PROCESADO, match[0])) as ds:
            if VAR_DIAG not in ds.variables: continue
            sensor = "S2A" if "S2A" in match[0] else "S2B" if "S2B" in match[0] else "S3"
            s_idx  = 0 if sensor == "S2A" else 1 if sensor == "S2B" else 0

            vd, np_, dm = extraer_inteligente(ds, row[COL_LAT], row[COL_LON], VAR_DIAG)
            if np_ > 0:
                res = {COL_SITIO: row[COL_SITIO], COL_FECHA: row[COL_FECHA],
                       COL_DATO: row[COL_DATO], "sensor": sensor,
                       VAR_DIAG.upper(): vd, "dist_shift_m": dm, "valid_pixels": np_}
                for cr, ondas in MAPEO_RRS.items():
                    vr = f"rrs_{ondas[s_idx]}"
                    if vr in ds.variables:
                        res[cr], _, _ = extraer_inteligente(ds, row[COL_LAT], row[COL_LON], vr)
                for algo in ALGOS_ACOLITE[1:]:
                    if algo in ds.variables:
                        res[algo.upper()], _, _ = extraer_inteligente(ds, row[COL_LAT], row[COL_LON], algo)
                matriz_val.append(res)
                print(f"  ✅ {row[COL_SITIO]} ({row[COL_FECHA]}): a {dm:.1f} m")
            else:
                print(f"  ☁️  {row[COL_SITIO]} ({row[COL_FECHA]}): sin agua en radio {RADIO_EXPANDIDO}px")
    except Exception as e:
        print(f"  ❌ {row[COL_SITIO]}: {e}")

pd.DataFrame(matriz_val).to_csv(ARCHIVO_VALIDACION, index=False)
print(f"\n✅ {ARCHIVO_VALIDACION}")

---
## 🔬 SECCIÓN 8 — Diagnóstico y Control de Calidad

Genera un reporte con el estado de extracción para cada punto de muestreo.

In [ ]:
# ── Diagnóstico espacial por estación ────────────────────────────────────────
import netCDF4 as nc, numpy as np, pandas as pd, os

# ✦ AUTO: Usa COL_* — funciona con cualquier CSV.
df_campo = pd.read_csv(ARCHIVO_CAMPO)
df_campo['date_std'] = pd.to_datetime(df_campo[COL_FECHA], errors='coerce').dt.strftime('%Y_%m_%d')
archivos = [f for f in os.listdir(CARPETA_PROCESADO) if f.endswith("_L2W.nc")]

# ✦ AUTO: Variable de diagnóstico = primer algoritmo en ALGOS_ACOLITE.
VAR_DIAG = ALGOS_ACOLITE[0]

reporte = []
print(f"🔎 Diagnóstico con variable: '{VAR_DIAG}'\n")

for _, row in df_campo.iterrows():
    match = [f for f in archivos if str(row['date_std']) in f]
    if not match:
        reporte.append({"Estacion": row[COL_SITIO], "Fecha": row[COL_FECHA],
                        COL_LAT: row[COL_LAT], COL_LON: row[COL_LON],
                        "Estado_Diagnostico": "Sin imagen en disco",
                        "Detalle": "Sin NetCDF para esta fecha."})
        continue
    try:
        with nc.Dataset(os.path.join(CARPETA_PROCESADO, match[0])) as ds:
            lats = ds.variables['lat'][:]
            lons = ds.variables['lon'][:]
            lat_p, lon_p = row[COL_LAT], row[COL_LON]

            if not (lats.min() <= lat_p <= lats.max() and lons.min() <= lon_p <= lons.max()):
                estado = "Fuera de imagen"
                motivo = "Coordenadas fuera del recorte ACOLITE (LIMIT_BOX)."
            else:
                dm = (lats-lat_p)**2 + (lons-lon_p)**2
                py, px = np.unravel_index(np.argmin(dm), dm.shape)
                if not coordenadas_validas(py, px, lats.shape):
                    estado = "Borde de imagen"
                    motivo = "Sin ventana 3×3 completa."
                elif VAR_DIAG in ds.variables:
                    v    = ds.variables[VAR_DIAG][py-1:py+2, px-1:px+2]
                    vals = v.compressed() if np.ma.is_masked(v) else v[~np.isnan(v)]
                    if vals.size > 0:
                        estado = "Agua (Válido)"
                        motivo = f"{vals.size}/9 píxeles válidos."
                    else:
                        estado = "Enmascarado (Nube/Tierra/Glint)"
                        motivo = "Todos los píxeles 3×3 enmascarados."
                else:
                    estado = "Error: variable ausente"
                    motivo = f"'{VAR_DIAG}' no encontrada en el NetCDF."

            reporte.append({"Estacion": row[COL_SITIO], "Fecha": row[COL_FECHA],
                            "Archivo": match[0], COL_LAT: lat_p, COL_LON: lon_p,
                            "Estado_Diagnostico": estado, "Detalle": motivo})
            if estado != "Agua (Válido)":
                print(f"  ⚠️  {row[COL_SITIO]} ({row[COL_FECHA]}) → {estado}")
    except Exception as e:
        print(f"  ❌ {match[0]}: {e}")

df_rep = pd.DataFrame(reporte)
df_rep.to_csv(ARCHIVO_REPORTE, index=False)
print(f"\n{'='*50}\n  RESUMEN\n{'='*50}")
print(df_rep['Estado_Diagnostico'].value_counts().to_string())
print(f"\n  💾 {ARCHIVO_REPORTE}")

---
## 🌿 SECCIÓN 9 — Estimación con Modelo MDN

Aplica el modelo MDN para estimar `MDN_PRODUCTO` desde las Rrs extraídas en §7.

> ⚠️ Requiere el entorno con MDN-STREAM instalado (ver §1).

In [ ]:
# ── 9.1 Fix de compatibilidad sklearn + carga de MDN ─────────────────────────
# MDN busca atributos 'unit_variance' y 'clip' en los escaladores de sklearn.
# Versiones antiguas no los tienen; este parche los agrega si faltan.
import sklearn.preprocessing
from sklearn.preprocessing import RobustScaler, MinMaxScaler

for cls in [RobustScaler, MinMaxScaler]:
    if not hasattr(cls, 'unit_variance'): setattr(cls, 'unit_variance', False)
    if not hasattr(cls, 'clip'):          setattr(cls, 'clip', False)

from MDN import get_sensor_bands, get_mdn_preds
from MDN.parameters import get_args

# ✦ AUTO: MDN_SENSOR y MDN_PRODUCTO vienen de la Sección 1.
kwargs = {'sensor': MDN_SENSOR, 'products': MDN_PRODUCTO}
# Incluir model_uid solo si se especificó en §1
if MDN_MODEL_UID:
    kwargs['model_uid'] = MDN_MODEL_UID

args      = get_args(kwargs, use_cmdline=False)
bands_mdn = get_sensor_bands(MDN_SENSOR)

print(f"✅ MDN cargado — Sensor: {MDN_SENSOR} | Producto: {MDN_PRODUCTO}")
print(f"   Bandas requeridas: {list(bands_mdn)} nm")

In [ ]:
# ── 9.2 Limpieza de Rrs y predicción MDN ─────────────────────────────────────
import pandas as pd, numpy as np

try:
    df = pd.read_csv(ARCHIVO_MATRIZ)
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Ejecuta §7 primero: {ARCHIVO_MATRIZ}")

# ✦ AUTO: Detectar columnas Rrs por patrón de nombre, no por nombre fijo.
#   Funciona con Rrs_443, rrs_443, Rrs443, etc.
cols_rrs = [c for c in df.columns if 'rrs' in c.lower()]
if not cols_rrs:
    raise ValueError(f"❌ Sin columnas Rrs. Columnas disponibles: {list(df.columns)}")
print(f"📡 Columnas Rrs ({len(cols_rrs)}): {cols_rrs}")

# Limpiar valores negativos (ACOLITE puede generarlos en píxeles problemáticos)
for col in cols_rrs:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        df.loc[df[col] < 0, col] = 0
        print(f"  ⚠️  {col}: {n_neg} negativos → 0")

print(f"\n  Mínimo Rrs post-limpieza: {df[cols_rrs].min().min():.6f}")

# ✦ AUTO: Mapear cada banda requerida por MDN a la columna Rrs más cercana.
#   Extrae el número de longitud de onda del nombre de columna y busca
#   la más cercana a cada banda requerida por el modelo.
x_test = np.zeros((len(df), len(bands_mdn)))
print(f"\n🗺️  Mapeo MDN → CSV:")
for i, wave in enumerate(bands_mdn):
    candidatos = []
    for c in cols_rrs:
        num = ''.join(filter(str.isdigit, c))
        if num:
            candidatos.append((abs(int(num) - wave), c))
    if not candidatos:
        print(f"  ⚠️  {wave} nm: sin columna disponible → ceros")
        continue
    col_elegida = sorted(candidatos)[0][1]
    x_test[:, i] = df[col_elegida].fillna(0).values
    print(f"  {wave} nm → '{col_elegida}'")

# Predicción
print(f"\n⚙️  Calculando {MDN_PRODUCTO.upper()}_MDN ({len(df)} muestras)...")
preds, desc = get_mdn_preds(x_test, args=args, model_type='testing')

# ✦ AUTO: Nombre de la columna de salida construido desde MDN_PRODUCTO.
COL_RESULTADO = f"{MDN_PRODUCTO.upper()}_MDN"
df[COL_RESULTADO] = preds['output'][:, desc[MDN_PRODUCTO]]

df.to_csv(ARCHIVO_FINAL, index=False)
print(f"\n✅ {COL_RESULTADO} guardado → {ARCHIVO_FINAL}")
print(f"\n📊 Estadísticas {COL_RESULTADO}:")
print(df[COL_RESULTADO].describe().to_string())

In [ ]:
# ── 9.3 Verificación final ────────────────────────────────────────────────────
import pandas as pd

df_check = pd.read_csv(ARCHIVO_FINAL)
# ✦ AUTO: Detectar columnas por patrón, no por nombre fijo.
cols_rrs = [c for c in df_check.columns if 'rrs' in c.lower()]
col_mdn  = f"{MDN_PRODUCTO.upper()}_MDN"

print("="*55)
print(f"  ✅ VERIFICACIÓN FINAL — {NOMBRE_PROYECTO}")
print("="*55)
print(f"  Registros          : {len(df_check)}")
print(f"  Columnas           : {len(df_check.columns)}")
print(f"  Bandas Rrs         : {len(cols_rrs)}")
if cols_rrs:
    print(f"  Rrs negativos      : {(df_check[cols_rrs] < 0).sum().sum()}")
    print(f"  Rrs mínimo         : {df_check[cols_rrs].min().min():.6f}")
if col_mdn in df_check.columns:
    s = df_check[col_mdn]
    print(f"  {col_mdn} no nulos: {s.notna().sum()} / {len(df_check)}")
    print(f"  {col_mdn} media   : {s.mean():.4f}")
    print(f"  {col_mdn} mediana : {s.median():.4f}")
    print(f"  {col_mdn} rango   : [{s.min():.4f}, {s.max():.4f}]")
print("\n" + "="*55)
print(f"  🎉 Pipeline '{NOMBRE_PROYECTO}' completado")
print("="*55)